# Pipeline (Colab) — Leitura + ML + “engenharia reversa” de corridas no Aeroporto Salgado Filho

Este notebook lê os Parquets salvos no padrão `_staging/batch=.../YYYY-MM-DD/<driver>/part-*.parquet`, consolida **fatos por janela de tempo** no aeroporto e faz o *join* com bases de **meteorologia** e **voos** para treinar um modelo e interpretar quais condições explicam corridas no aeroporto.

> ⚠️ Ajuste os nomes das colunas (schema) do seu `trips_log`, `weather` e `flights` nas seções indicadas.


## 0) Instalar dependências e imports


In [ ]:
!pip -q install pyarrow polars scikit-learn

import os
from pathlib import Path
import re
import numpy as np
import pandas as pd
import polars as pl
import pyarrow.dataset as ds

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.ensemble import HistGradientBoostingClassifier


## 1) Configurações de caminho e inspeção do formato salvo


In [ ]:
# === AJUSTE AQUI ===
BASE = Path("/content/drive/MyDrive/DOUTORADO/003_DADOS_SINTETICOS/outputs_simulation_v5_optimized")

TRIPS_STAGING = BASE / "trips_log" / "_staging"
TRIPS_FINAL   = BASE / "trips_log" / "monthly"   # saída consolidada (fatos por mês)

WEATHER_CSV   = Path("/content/Weather_enhanced.csv")
FLIGHTS_CSV   = Path("/content/Flights_enhanced.csv")  # ajuste para sua base real

TRIPS_FINAL.mkdir(parents=True, exist_ok=True)

def list_some_parquets(root: Path, limit=20):
    files = list(root.rglob("*.parquet"))
    print("Total parquet files:", len(files))
    for p in files[:limit]:
        print(p)
    return files

all_files = list_some_parquets(TRIPS_STAGING, limit=10)


Total parquet files: 0


## 2) Definir o aeroporto (H3) e granularidade temporal


In [ ]:
# === AJUSTE AQUI: lista de H3 do aeroporto (mesma resolução do campo que você vai filtrar no trips e no weather) ===
AIRPORT_H3 = set([
    # exemplo (substitua pela sua lista completa)
    '86a901287ffffff','86a90128fffffff','86a90129fffffff','86a90e937ffffff',
    '86a90166fffffff','86a901297ffffff','86a9012afffffff','86a9012a7ffffff',
    '86a9012b7ffffff','86a901667ffffff','86a90174fffffff','86a90e907ffffff',
    '86a90e92fffffff','86a9012d7ffffff','86a90164fffffff','86a90e917ffffff',
    '86a90e927ffffff'
])

# janela temporal de agregação para ML
TIME_BIN_MINUTES = 5


## 3) Utilitários: parsing do path e binning temporal


In [ ]:
DAY_RE   = re.compile(r"/(\d{4}-\d{2}-\d{2})/")
BATCH_RE = re.compile(r"/batch=(\d+)/")
DRIVER_RE = re.compile(r"/([^/]+)/part-")  # pega o folder imediatamente antes de part-*.parquet

def parse_path_metadata(path: str):
    day   = DAY_RE.search(path)
    batch = BATCH_RE.search(path)
    driver = DRIVER_RE.search(path)
    return {
        "day": day.group(1) if day else None,
        "batch": batch.group(1) if batch else None,
        "driver_folder": driver.group(1) if driver else None,
    }

def to_time_bin(ts: pl.Expr, minutes=5):
    return ts.dt.truncate(f"{minutes}m")

def month_str_from_day(day: str) -> str:
    return day[:7]  # YYYY-MM


## 4) Construir fatos de corridas no aeroporto a partir do staging


In [ ]:
def scan_trips_parquets(parquet_paths):
    return pl.scan_parquet([str(p) for p in parquet_paths])

def build_airport_facts_from_paths(parquet_paths, airport_h3: set, minutes=5):
    lf = scan_trips_parquets(parquet_paths)
    cols = lf.columns

    # === AJUSTE AQUI: nomes de colunas do trips ===
    # 1) timestamp
    time_col = "time_utc" if "time_utc" in cols else ("timestamp" if "timestamp" in cols else None)

    # 2) origem (h3) — ajuste para seu schema real
    origin_col = "origin" if "origin" in cols else ("origin_h3" if "origin_h3" in cols else None)

    if time_col is None or origin_col is None:
        raise ValueError(f"Colunas esperadas não encontradas. Colunas disponíveis (amostra): {cols[:80]}")

    # filtro cedo: só origens no aeroporto
    lf = lf.filter(pl.col(origin_col).is_in(list(airport_h3)))

    # normaliza timestamp e cria bin
    lf = lf.with_columns([
        pl.col(time_col).cast(pl.Datetime).alias("time_utc"),
        to_time_bin(pl.col(time_col).cast(pl.Datetime), minutes).alias("time_bin")
    ])

    # agrega por bin
    aggs = [
        pl.len().alias("rides_airport"),
        pl.col(origin_col).n_unique().alias("n_unique_h3_origin"),
    ]
    if "driver_id" in cols:
        aggs.append(pl.col("driver_id").n_unique().alias("n_unique_drivers"))
    else:
        aggs.append(pl.lit(None).alias("n_unique_drivers"))

    facts = (
        lf.group_by("time_bin")
          .agg(aggs)
          .sort("time_bin")
    )

    return facts.collect(streaming=True)

def group_paths_by_month(parquet_paths):
    by_month = {}
    for p in parquet_paths:
        meta = parse_path_metadata(str(p))
        if meta["day"] is None:
            continue
        m = month_str_from_day(meta["day"])
        by_month.setdefault(m, []).append(p)
    return by_month

by_month = group_paths_by_month(all_files)
print("Meses encontrados:", sorted(by_month.keys())[:12], " | total:", len(by_month))


### 4.1) Gerar e salvar fatos mensais (Parquet) — recomendado para ML


In [ ]:
# Isso gera datasets MUITO menores (ex.: bins de 5 min => ~8.6k linhas por mês)
for m, paths in sorted(by_month.items()):
    out = TRIPS_FINAL / f"airport_facts_{m}.parquet"
    if out.exists():
        print("Skip (exists):", out.name)
        continue

    print("Building:", m, "files:", len(paths))
    facts_m = build_airport_facts_from_paths(paths, AIRPORT_H3, minutes=TIME_BIN_MINUTES)
    facts_m.write_parquet(str(out))
    print("Saved:", out)


## 5) Preparar meteorologia (weather) para join por time_bin


In [ ]:
w = pl.read_csv(str(WEATHER_CSV))

# === AJUSTE AQUI: nomes de colunas do weather ===
time_w = "time_utc" if "time_utc" in w.columns else "timestamp"
loc_w  = "location" if "location" in w.columns else "h3_index"

w = w.with_columns([
    pl.col(time_w).str.strptime(pl.Datetime, strict=False).alias("time_utc")
])

# filtro para aeroporto (mesma resolução do H3)
w_airport = w.filter(pl.col(loc_w).is_in(list(AIRPORT_H3)))

# agrega no mesmo bin do trips
w_airport = (
    w_airport
    .with_columns([to_time_bin(pl.col("time_utc"), TIME_BIN_MINUTES).alias("time_bin")])
    .group_by("time_bin")
    .agg([
        (pl.mean("temperature") if "temperature" in w_airport.columns else pl.lit(None)).alias("temp_mean"),
        (pl.mean("relative_humidity_2m") if "relative_humidity_2m" in w_airport.columns else pl.lit(None)).alias("rh_mean"),
        # adicione variáveis que existirem, por exemplo:
        # (pl.mean("precipitation") if "precipitation" in w_airport.columns else pl.lit(None)).alias("precip_mean"),
        # (pl.mean("wind_speed") if "wind_speed" in w_airport.columns else pl.lit(None)).alias("wind_mean"),
    ])
    .sort("time_bin")
)

w_airport.head()


## 6) Preparar voos (flights) para join por time_bin + lags/rolling


In [ ]:
f = pl.read_csv(str(FLIGHTS_CSV))

# === AJUSTE AQUI: nomes de colunas do flights ===
time_f = "event_time_utc" if "event_time_utc" in f.columns else "time_utc"
type_f = "event_type" if "event_type" in f.columns else "type"  # arrival/departure

f = f.with_columns([
    pl.col(time_f).str.strptime(pl.Datetime, strict=False).alias("event_time_utc"),
    to_time_bin(pl.col(time_f).str.strptime(pl.Datetime, strict=False), TIME_BIN_MINUTES).alias("time_bin")
])

# contagens por bin e tipo (pivot)
fl_bins = (
    f.group_by(["time_bin", type_f])
     .agg(pl.len().alias("n_flights"))
     .pivot(values="n_flights", index="time_bin", columns=type_f)
     .fill_null(0)
     .sort("time_bin")
)

# renomeia colunas
for c in fl_bins.columns:
    if c != "time_bin":
        fl_bins = fl_bins.rename({c: f"fl_{c}"})

def add_lag_and_rolling(df: pl.DataFrame, cols, bins_per_window=6):
    out = df.sort("time_bin")
    for c in cols:
        out = out.with_columns([
            pl.col(c).shift(1).alias(f"{c}_lag1"),
            pl.col(c).rolling_sum(bins_per_window).alias(f"{c}_sum_last_{bins_per_window}bins"),
        ])
    return out

flight_cols = [c for c in fl_bins.columns if c != "time_bin"]
fl_feats = add_lag_and_rolling(fl_bins, flight_cols, bins_per_window=6)  # 30 min se bin=5 min

fl_feats.head()


## 7) Montar dataset final (fatos + weather + flights) e criar target


In [ ]:
facts_files = sorted(TRIPS_FINAL.glob("airport_facts_*.parquet"))
facts = pl.concat([pl.read_parquet(str(p)) for p in facts_files]).sort("time_bin")

df = (
    facts
    .join(w_airport, on="time_bin", how="left")
    .join(fl_feats, on="time_bin", how="left")
    .fill_null(0)
)

# target binário: houve corrida no aeroporto no bin?
df = df.with_columns([
    (pl.col("rides_airport") > 0).cast(pl.Int8).alias("y_has_ride")
])

# features temporais
df = df.with_columns([
    pl.col("time_bin").dt.hour().alias("hour"),
    pl.col("time_bin").dt.weekday().alias("weekday"),
    pl.col("time_bin").dt.month().alias("month"),
])

df.select(["time_bin","rides_airport","y_has_ride","hour","weekday","temp_mean","rh_mean"]).head(10)


## 8) Treinar ML com split temporal (sem vazamento) e avaliar


In [ ]:
pdf = df.to_pandas().sort_values("time_bin")

target = "y_has_ride"
drop_cols = ["time_bin", "rides_airport", "y_has_ride"]
feature_cols = [c for c in pdf.columns if c not in drop_cols]

X = pdf[feature_cols].values
y = pdf[target].values

tscv = TimeSeriesSplit(n_splits=5)
aucs, aps = [], []

for fold, (tr, te) in enumerate(tscv.split(X), 1):
    model = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.05, max_iter=400)
    model.fit(X[tr], y[tr])

    proba = model.predict_proba(X[te])[:, 1]
    auc = roc_auc_score(y[te], proba)
    ap = average_precision_score(y[te], proba)

    aucs.append(auc); aps.append(ap)
    print(f"Fold {fold}: AUC={auc:.3f}  AP={ap:.3f}")

print("AUC mean:", float(np.mean(aucs)), "AP mean:", float(np.mean(aps)))


## 9) Engenharia reversa: importância de variáveis (Permutation Importance) + PDP


In [ ]:
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
import matplotlib.pyplot as plt

# holdout temporal simples (80/20)
split = int(len(pdf) * 0.8)
X_tr, X_te = X[:split], X[split:]
y_tr, y_te = y[:split], y[split:]

final_model = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.05, max_iter=500)
final_model.fit(X_tr, y_tr)

proba = final_model.predict_proba(X_te)[:, 1]
print("Holdout AUC:", roc_auc_score(y_te, proba), "AP:", average_precision_score(y_te, proba))

perm = permutation_importance(final_model, X_te, y_te, n_repeats=10, random_state=42)
imp = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

display(imp.head(25))

top_feats = imp["feature"].head(6).tolist()
idxs = [feature_cols.index(f) for f in top_feats[:3]]

fig = plt.figure(figsize=(10, 6))
PartialDependenceDisplay.from_estimator(final_model, X_te, features=idxs)
plt.show()


## 10) Checklist rápido para tornar o estudo defensável


- **Consistência de timezone**: mantenha tudo em UTC ou tudo em `America/Sao_Paulo`.
- **Evite leakage**: não use variáveis que dependem de “futuro real” (a menos que seja *schedule* conhecido).
- **Granularidade**: 5 min é sensível e esparso; 15 min é mais estável e interpretável.
- **Explicação**: use importâncias, PDP e/ou um modelo “explicativo” (árvore rasa) com top features.
